In [0]:
%sql
TRUNCATE TABLE workspace.silver.dim_product_tmp;

INSERT INTO workspace.silver.dim_product_tmp (
    Material,
    MaterialByCustomer,
    MaterialGroup,
    MaterialPricingGroup,
    OriginallyRequestedMaterial,
    _rescued_data,
    _ingestion_timestamp,
    Status_Cleansing,
    Status_DQ1,
    Status_DQ2,
    Status_DQ3,
    Status_DQ4,
    Status_DQ5,
    Status_Process
)
SELECT DISTINCT
    Material,
    MaterialByCustomer,
    MaterialGroup,
    MaterialPricingGroup,
    OriginallyRequestedMaterial,
    _rescued_data,
    _ingestion_timestamp,
    'PENDING' AS Status_Cleansing,
    'PENDING' AS Status_DQ1,
    'PENDING' AS Status_DQ2,
    'PENDING' AS Status_DQ3,
    'PENDING' AS Status_DQ4,
    'PENDING' AS Status_DQ5,
    'IN_PROGRESS' AS Status_Process
FROM workspace.bronze.sales_order_item;

In [0]:
%sql
UPDATE workspace.silver.dim_product_tmp
SET 
    Material = CASE 
        WHEN regexp_replace(Material, '^0+', '') = '' AND Material IS NOT NULL THEN '0'
        ELSE coalesce(regexp_replace(Material, '^0+', ''), Material)
    END,
    OriginallyRequestedMaterial = CASE 
        WHEN regexp_replace(OriginallyRequestedMaterial, '^0+', '') = '' AND OriginallyRequestedMaterial IS NOT NULL THEN '0'
        ELSE coalesce(regexp_replace(OriginallyRequestedMaterial, '^0+', ''), OriginallyRequestedMaterial)
    END,
    MaterialByCustomer = trim(upper(MaterialByCustomer)),
    MaterialGroup = trim(upper(MaterialGroup)),
    MaterialPricingGroup = trim(upper(MaterialPricingGroup)),
    Status_Cleansing = 'COMPLETED'
WHERE Status_Cleansing = 'PENDING' 
  AND Status_Process = 'IN_PROGRESS';

In [0]:
%sql
UPDATE workspace.silver.dim_product_tmp
SET 
    Status_DQ1 = CASE 
        WHEN Material IS NULL OR trim(Material) = '' THEN 'FAILED'
        ELSE 'PASSED'
    END,
    Status_Process = CASE 
        WHEN Material IS NULL OR trim(Material) = '' THEN 'QUARANTINED'
        ELSE Status_Process
    END
WHERE Status_Cleansing = 'COMPLETED' 
  AND Status_Process = 'IN_PROGRESS';

In [0]:
%sql
MERGE INTO workspace.silver.dim_product_tmp AS tgt
USING (
    SELECT 
        Material,
        COUNT(DISTINCT named_struct(
            'MaterialByCustomer', MaterialByCustomer,
            'MaterialGroup', MaterialGroup,
            'MaterialPricingGroup', MaterialPricingGroup,
            'OriginallyRequestedMaterial', OriginallyRequestedMaterial
        )) AS conflict_count
    FROM workspace.silver.dim_product_tmp
    WHERE Status_Process = 'IN_PROGRESS'
    GROUP BY Material
) AS src
ON tgt.Material = src.Material
WHEN MATCHED AND tgt.Status_Process = 'IN_PROGRESS' AND tgt.Status_DQ1 = 'PASSED' THEN
    UPDATE SET 
        tgt.Status_DQ2 = CASE 
            WHEN src.conflict_count > 1 THEN 'FAILED' 
            ELSE 'PASSED' 
        END,
        tgt.Status_Process = CASE 
            WHEN src.conflict_count > 1 THEN 'QUARANTINED' 
            ELSE tgt.Status_Process 
        END;

In [0]:
%sql
UPDATE workspace.silver.dim_product_tmp
SET 
    Status_DQ3 = CASE 
        WHEN (Material LIKE '0%' AND Material <> '0') 
          OR (OriginallyRequestedMaterial LIKE '0%' AND OriginallyRequestedMaterial <> '0') THEN 'FAILED'
        ELSE 'PASSED'
    END,
    Status_Process = CASE 
        WHEN (Material LIKE '0%' AND Material <> '0') 
          OR (OriginallyRequestedMaterial LIKE '0%' AND OriginallyRequestedMaterial <> '0') THEN 'QUARANTINED'
        ELSE Status_Process
    END
WHERE Status_Process = 'IN_PROGRESS'
  AND Status_DQ2 = 'PASSED';

In [0]:
%sql
UPDATE workspace.silver.dim_product_tmp
SET 
    Status_DQ4 = CASE 
        WHEN length(coalesce(MaterialGroup, '')) > 9 
          OR length(coalesce(MaterialPricingGroup, '')) > 4 THEN 'FAILED'
        ELSE 'PASSED'
    END,
    Status_Process = CASE 
        WHEN length(coalesce(MaterialGroup, '')) > 9 
          OR length(coalesce(MaterialPricingGroup, '')) > 4 THEN 'QUARANTINED'
        ELSE Status_Process
    END
WHERE Status_Process = 'IN_PROGRESS'
  AND Status_DQ3 = 'PASSED';

In [0]:
%sql
UPDATE workspace.silver.dim_product_tmp
SET 
    Status_DQ5 = CASE 
        WHEN _rescued_data IS NOT NULL THEN 'FAILED'
        ELSE 'PASSED'
    END,
    Status_Process = CASE 
        WHEN _rescued_data IS NOT NULL THEN 'QUARANTINED'
        ELSE Status_Process
    END
WHERE Status_Process = 'IN_PROGRESS'
  AND Status_DQ4 = 'PASSED';

In [0]:
%sql
-- Sweeper: Consolidated routing of rejected records to Quarantine
INSERT INTO workspace.silver.sales_order_quarantine (
    SourceEntity,
    RecordIdentifier,
    FailedRule,
    FailureSeverity,
    RawRecord,
    IngestionTimestamp
)
SELECT 
    'DIM_PRODUCT' AS SourceEntity,
    coalesce(Material, 'UNKNOWN_KEY') AS RecordIdentifier,
    CASE 
        WHEN Status_DQ1 = 'FAILED' THEN 'DQ1_NOT_NULL_VIOLATION'
        WHEN Status_DQ2 = 'FAILED' THEN 'DQ2_ATTRIBUTE_CONFLICT'
        WHEN Status_DQ3 = 'FAILED' THEN 'DQ3_ALPHA_CONVERSION_ERROR'
        WHEN Status_DQ4 = 'FAILED' THEN 'DQ4_DOMAIN_LENGTH_EXCEEDED'
        WHEN Status_DQ5 = 'FAILED' THEN 'DQ5_RESCUED_DATA_CORRUPT'
        ELSE 'UNKNOWN_FAILURE'
    END AS FailedRule,
    'HARD' AS FailureSeverity,
    to_json(named_struct(
        'Material', Material,
        'MaterialByCustomer', MaterialByCustomer,
        'MaterialGroup', MaterialGroup,
        'MaterialPricingGroup', MaterialPricingGroup,
        'OriginallyRequestedMaterial', OriginallyRequestedMaterial,
        '_rescued_data', _rescued_data
    )) AS RawRecord,
    current_timestamp() AS IngestionTimestamp
FROM workspace.silver.dim_product_tmp
WHERE Status_Process = 'QUARANTINED';

In [0]:
%sql
UPDATE workspace.silver.dim_product_tmp
SET Status_Process = 'READY_FOR_STG'
WHERE Status_Process = 'IN_PROGRESS'
  AND Status_DQ1 = 'PASSED'
  AND Status_DQ2 = 'PASSED'
  AND Status_DQ3 = 'PASSED'
  AND Status_DQ4 = 'PASSED'
  AND Status_DQ5 = 'PASSED';

In [0]:
%sql
-- Promotion: Load and idempotency into target dimensión
MERGE INTO workspace.silver.dim_product_stg AS tgt
USING (
    SELECT 
        Material,
        MaterialByCustomer,
        MaterialGroup,
        MaterialPricingGroup,
        OriginallyRequestedMaterial,
        min(_ingestion_timestamp) AS ValidFrom
    FROM workspace.silver.dim_product_tmp
    WHERE Status_Process = 'READY_FOR_STG'
    GROUP BY 
        Material,
        MaterialByCustomer,
        MaterialGroup,
        MaterialPricingGroup,
        OriginallyRequestedMaterial
) AS src
ON tgt.Material = src.Material AND tgt.IsCurrent = TRUE
WHEN MATCHED AND (
    tgt.MaterialByCustomer <=> src.MaterialByCustomer = FALSE OR
    tgt.MaterialGroup <=> src.MaterialGroup = FALSE OR
    tgt.MaterialPricingGroup <=> src.MaterialPricingGroup = FALSE OR
    tgt.OriginallyRequestedMaterial <=> src.OriginallyRequestedMaterial = FALSE
) THEN
    -- If attributes changed, close previous versión
    UPDATE SET 
        tgt.ValidTo = current_timestamp(),
        tgt.IsCurrent = FALSE
WHEN NOT MATCHED THEN
    -- Insert new material
    INSERT (
        Material,
        MaterialByCustomer,
        MaterialGroup,
        MaterialPricingGroup,
        OriginallyRequestedMaterial,
        ValidFrom,
        ValidTo,
        IsCurrent
    )
    VALUES (
        src.Material,
        src.MaterialByCustomer,
        src.MaterialGroup,
        src.MaterialPricingGroup,
        src.OriginallyRequestedMaterial,
        coalesce(src.ValidFrom, current_timestamp()),
        NULL,
        TRUE
    );